# Show4DSTEM

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bobleesj/quantem.widget/blob/main/docs/tutorials/show4dstem.ipynb)

`Show4DSTEM` opens a 4D-STEM dataset with live virtual detectors: a bright-field / annular-dark-field aperture over the diffraction stack on one side, the resulting virtual image on the other. It accepts a NumPy array, a PyTorch tensor, a quantem `Dataset4dstem`, or the output of `load(...)`.

This tutorial uses a small synthetic 4D-STEM scan so it runs anywhere, including Colab and the documentation build environment.

```{note}
Exported with `offline=True`: a compact detector stack ships to the browser and the virtual-detector math runs in **WebGPU**, so the widget below is fully interactive with no Python kernel. Drag the bright-field disk or change the detector radius and the virtual image recomputes in the browser. Large datasets or full uint16-count workflows keep the CUDA/MPS kernel path.
```

In [1]:
import numpy as np
from quantem.core.datastructures import Dataset4dstem
from quantem.widget import Show4DSTEM

scan_y, scan_x = np.indices((48, 48), dtype=np.float32)
det_y, det_x = np.indices((24, 24), dtype=np.float32)

# A compact synthetic 4D-STEM dataset: each probe position has a diffraction
# disk whose center shifts across the scan, plus a weak annular background.
center_y = 12 + 2.0 * np.sin(scan_y / 8) + 0.8 * np.cos(scan_x / 9)
center_x = 12 + 2.0 * np.cos(scan_x / 7) - 0.8 * np.sin(scan_y / 10)
r2 = (det_y[None, None] - center_y[..., None, None]) ** 2 + (det_x[None, None] - center_x[..., None, None]) ** 2
ring = np.exp(-((np.sqrt((det_y - 12) ** 2 + (det_x - 12) ** 2) - 7) ** 2) / 8)
data = (180 * np.exp(-r2 / 12) + 25 * ring[None, None]).astype(np.float32)

stem_dataset = Dataset4dstem.from_array(
    data,
    sampling=(0.42, 0.42, 0.75, 0.75),
    units=("nm", "nm", "mrad", "mrad"),
    name="Synthetic 4D-STEM scan",
)
stem_dataset.array.shape

(48, 48, 24, 24)

## Virtual detectors

The synthetic 4D-STEM scan is wrapped in a quantem `Dataset4dstem`, so scan and detector calibration travel with the data. Drag the bright-field disk across the diffraction pattern, or use the BF / ABF / ADF presets, and watch the virtual image update live.

In [2]:
Show4DSTEM(stem_dataset, offline=True)

  to mps: 0.01s (0.0 GB)


  auto_detect_center: 0.10s
  virtual image + frame: 0.00s
Show4DSTEM ready in 0.30s
  shape   : 48x48x24x24
  backend : Apple GPU (Metal, torch)   device=mps
  data in : Apple unified memory   (NumPy input uploaded to device)


Show4DSTEM(shape=(48, 48, 24, 24), sampling=(0.42 nm, 0.75 mrad), pos=(24, 24), title='Synthetic 4D-STEM scan')